## 1. Runtime Parameters and Environment Configuration


In [0]:
# Create Databricks widgets for environment and pipeline name
dbutils.widgets.text("env", "")
dbutils.widgets.text("pipeline_name", "")


In [0]:
# Retrieve widget values for environment and pipeline name
env = dbutils.widgets.get("env")
pipeline_name = dbutils.widgets.get("pipeline_name")


In [0]:
# Validate pipline name is provided
if not pipeline_name:
    raise ValueError("pipeline_name parameter is mandatory")


In [0]:
# Define configuration for different environments and validate input
config = {
    "dev": {
        "storage_account": "carsalesproject23",
        "bronze_container": "bronze",
        "silver_container": "silver",
        "jdbc": {
            "url": "jdbc:sqlserver://car-sales-server.database.windows.net:1433;database=car_sales_db;encrypt=true;trustServerCertificate=false;hostNameInCertificate=*.database.windows.net;loginTimeout=30;"
        }
    },
    "prod": {
        "storage_account": "carsalesprod",
        "bronze_container": "bronze",
        "silver_container": "silver",
         "jdbc": {
            "url": "jdbc:sqlserver://car-sales-server.database.windows.net:1433;database=car_sales_db;encrypt=true;trustServerCertificate=false;hostNameInCertificate=*.database.windows.net;loginTimeout=30;"
        }
    }
}
if env not in config:
    raise ValueError(f"Invalid env '{env}'. Expected one of {list(config.keys())}")

cfg = config[env]


In [0]:
# Retrieve SQL Server credentials from Databricks secrets
sql_password = dbutils.secrets.get("kv-carsales", "sql-password")
sql_user = dbutils.secrets.get("kv-carsales", "sql-user")

In [0]:
# Import necessary PySpark functions and types
from pyspark.sql.functions import *
from pyspark.sql.types import *

## 2. Identify New Successful Bronze Runs


In [0]:
# Read pipeline run control table and filter for successful runs for the current pipeline
run_df = spark.read \
    .format("jdbc") \
    .option("url", cfg["jdbc"]["url"]) \
    .option("dbtable", "pipeline_run_control") \
    .option("user", sql_user) \
    .option("password",sql_password) \
    .load()

success_runs = run_df.filter(
    (col("pipeline_name") == pipeline_name) &
    (col("run_status") == "SUCCESS")
).select("run_id", "bronze_path")

In [0]:
# Define base paths for silver and bronze storage containers
silver_base_path = f"abfss://{cfg['silver_container']}@{cfg['storage_account']}.dfs.core.windows.net"
bronze_base_path = f"abfss://{cfg['bronze_container']}@{cfg['storage_account']}.dfs.core.windows.net"

In [0]:
#Create processed runs tracking table if not exists and identify new runs to process
spark.sql(f"""
CREATE TABLE IF NOT EXISTS cars_catalog.silver.silver_processed_runs (
    run_id STRING,
    processed_ts TIMESTAMP
)
USING DELTA
LOCATION '{silver_base_path}/_control/silver_processed_runs'
""")
processed_runs = spark.sql("""
                         select * from cars_catalog.silver.silver_processed_runs
                         """)
new_runs = success_runs.join(
    processed_runs,
    success_runs.run_id == processed_runs.run_id,
    "left_anti"
)

In [0]:
#Build list of new bronze paths to process, exit if none
bronze_paths = [
    f"{bronze_base_path}/{row.bronze_path}"
    for row in new_runs.select("bronze_path").collect()
]
bronze_paths = [
    f"{bronze_base_path}/{row.bronze_path}"
    for row in new_runs.select("bronze_path").toLocalIterator()
]

if not bronze_paths:
    print("No new Bronze runs to process")
    dbutils.notebook.exit("No-op")



## 3. Read Bronze Data with Fixed Schema


In [0]:
# Define schema and read sales data from bronze paths
# Ensure no schema drift
sales_schema = StructType([
    StructField("Branch_ID", StringType(), True),
    StructField("Dealer_ID", StringType(), True),
    StructField("Model_ID", StringType(), True),
    StructField("Revenue", LongType(), True),
    StructField("Units_Sold", LongType(), True),
    StructField("Date_ID", StringType(), True),
    StructField("Day", IntegerType(), True),
    StructField("Month", IntegerType(), True),
    StructField("Year", IntegerType(), True),
    StructField("BranchName", StringType(), True),
    StructField("DealerName", StringType(), True),
    StructField("Product_Name", StringType(), True)
])

sales_data_df = spark.read.format("parquet") \
                           .schema(sales_schema) \
                           .load(*bronze_paths) 

## 4. Silver Transformations and Data Quality Rules


In [0]:
#Add model_category column by splitting Model_ID
sales_data_df = sales_data_df.withColumn('model_category', split('Model_ID', '-')[0])

In [0]:
# Add revenue per unit column, handling division by zero
sales_data_df = sales_data_df.withColumn("rev_per_unit",
    when(col("Units_Sold") == 0, None)
    .otherwise(round(col("Revenue") / col("Units_Sold"), 2)))

In [0]:
# Add data quality error column based on business rules
dq_df = sales_data_df.withColumn(
    "dq_error",
    concat_ws(
        "|",
        when(col("Branch_ID").isNull(), lit("branch_id_null")),
        when(col("Dealer_ID").isNull(), lit("dealer_id_null")),
        when(col("Model_ID").isNull(), lit("model_id_null")),
        when(col("Date_ID").isNull(), lit("date_id_null")),
        when(col("Revenue") < 0, lit("negative_revenue")),
        when(col("Units_Sold") < 0, lit("negative_units")),
        when((col("Month") < 1) | (col("Month") > 12), lit("invalid_month")),
        when(col("Year") > year(current_date()), lit("future_year"))
    )
)


In [0]:
# Set dq_error to None if no errors found
dq_df = dq_df.withColumn(
    "dq_error",
    when(col("dq_error") == "", None).otherwise(col("dq_error"))
)

In [0]:
# Split data into good and bad records based on dq_error
sales_good_data_df = dq_df.filter(col("dq_error").isNull()).drop(col('dq_error'))
sales_bad_data_df = dq_df.filter(col("dq_error").isNotNull()).drop(col('dq_error'))

In [0]:
# Clean and standardize good sales data
clean_sales_df = sales_good_data_df \
    .withColumn("BranchName", trim(col("BranchName"))) \
    .withColumn("DealerName", trim(col("DealerName"))) \
    .withColumn("Product_Name", trim(col("Product_Name"))) \
    .withColumn("model_category", lower(trim(col("model_category"))))

## 5. Write Silver Output and Mark Runs as Processed


In [0]:
# Write clean and bad sales data to silver and quarantine locations
clean_sales_df.write.format('parquet') \
        .mode('append') \
        .option('path',f'{silver_base_path}/carsales').save()

sales_bad_data_df.write.format('parquet') \
        .mode('append') \
        .option('path',f'{silver_base_path}/quarantine/sales_bad_records').save()       

In [0]:
# Insert new runs processed in the tracking table
new_runs.select(
    col("run_id"),
    current_timestamp().alias("processed_ts")
).write \
 .format("delta") \
 .mode("append") \
 .saveAsTable("cars_catalog.silver.silver_processed_runs")
